# 地方競馬 〜 ワイドオッズのページの下調べ（アクセスは最大3回）

本物のワイドオッズを使えるかを確かめるため、**1レース分のワイドオッズのページを1回だけ取得**して保存する。
1. robots.txt を確認（1回）。ワイドオッズのページが禁止されていれば、そこで止まる
2. 対象のレースを決める（`RACE_ID` を空にすると、今日の開催一覧を1回だけ取得して最初のレースを使う）
3. ワイドオッズのページを1回取得して保存し、オッズがページの中に書かれているかを確認する

- 間隔は 3.5〜4.5秒、1本だけ。403・429・503 が返ったら**その場で止まる**（再試行しない）
- **Internet を ON** にして Run All。1分以内に終わる
- 終わったら Output の `probe_wide.html` と、このノートブックのログを送ってください（ページの形に合わせて読み取りの部品を作ります）

In [ ]:
RACE_ID = ""          # 例 "202644111612"。空なら今日の開催一覧から最初のレース
import requests, time, random, re, datetime as dt, urllib.robotparser
BASE = "https://nar.netkeiba.com"
UA = "Mozilla/5.0 (compatible; personal-research; low-rate single-thread)"
JST = dt.timezone(dt.timedelta(hours=9))
S = requests.Session(); S.headers.update({"User-Agent": UA, "Accept-Language": "ja"})
_last = [0.0]; N = [0]
def get(url, allow_404=False):
    w = 3.5 + random.uniform(0, 1.0) - (time.time() - _last[0])
    if w > 0: time.sleep(w)
    r = S.get(url, timeout=(10, 20)); _last[0] = time.time(); N[0] += 1
    print(f"  GET {url} → {r.status_code}（{N[0]}回目）")
    if r.status_code in (403, 429, 503):
        raise SystemExit(f"HTTP {r.status_code}: アクセス制限の可能性があるため、ここで止めます（再試行しません）")
    if allow_404 and r.status_code == 404:
        return r
    r.raise_for_status()
    return r
def decode(r):
    for enc in ["euc-jp", "utf-8", "cp932"]:
        try:
            return r.content.decode(enc)
        except UnicodeDecodeError:
            pass
    return r.content.decode("euc-jp", "replace")

rb = get(BASE + "/robots.txt", allow_404=True)
rp = urllib.robotparser.RobotFileParser()
if rb.status_code == 404:
    rp.parse([])            # robots.txt が無い＝クロールの制限なし（利用規約は別途守る）
    print("robots.txt は置かれていません（404）。制限なしとして扱います。利用規約の範囲（私的利用・低頻度・1本だけ）は引き続き守ります")
else:
    rp.parse(rb.text.splitlines())
    print("robots.txt:\n" + rb.text[:1500])

In [ ]:
if not RACE_ID:
    d = dt.datetime.now(JST).strftime("%Y%m%d")
    lu = f"{BASE}/top/race_list_sub.html?kaisai_date={d}"
    if not rp.can_fetch(UA, lu):
        raise SystemExit("開催一覧のページが robots.txt で禁止されているため止めます")
    ids = re.findall(r"race_id=(\d{12})", decode(get(lu)))
    if not ids:
        raise SystemExit("今日の開催が見つかりません。RACE_ID に race_id を入れて再実行してください")
    RACE_ID = ids[0]
print("対象レース:", RACE_ID)
WIDE_URL = f"{BASE}/odds/?type=b5&race_id={RACE_ID}&housiki=c99"   # 人気順の一覧（全組・下限と上限）
print("ワイドオッズのページ:", WIDE_URL, "/ robots.txt で", "許可" if rp.can_fetch(UA, WIDE_URL) else "禁止")
if not rp.can_fetch(UA, WIDE_URL):
    raise SystemExit("ワイドオッズのページが robots.txt で禁止されているため、取得しません（この方法は使えません）")

In [ ]:
h = decode(get(WIDE_URL))
open("/kaggle/working/probe_wide.html", "w", encoding="utf-8").write(h)
print(f"保存しました: probe_wide.html（{len(h):,}文字）/ アクセス合計 {N[0]}回")
cells_ = re.findall(r'<td[^>]*class="[^"]*Odds[^"]*"[^>]*>(.*?)</td>', h, re.S)
vals = [re.sub(r"<[^>]+>", "", c).strip() for c in cells_]
print("Odds の欄:", len(vals), "例:", vals[:12])
txt = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", h))
rng = re.findall(r"(\d+) (\d+) (\d+\.\d) (\d+\.\d)", txt)
print("「組 下限 上限」の並び:", len(rng), "例:", rng[:6])
for m in re.finditer(r"<tr[^>]*>(?:(?!</tr>).)*?\d+\.\d(?:(?!</tr>).)*?</tr>", h, re.S):
    print("表の1行の例（HTML）:", m.group(0)[:600].replace("\n", " ")); break
for kw in ["api_get", "odds_get", "ajax", "jsonp", "Odds_Ninki", "Wide"]:
    m = [x.start() for x in re.finditer(kw, h)][:2]
    for i in m:
        print(f"[{kw}]", h[max(0, i - 120):i + 200].replace("\n", " "))